# 01. Data Preparation and Preprocessing Pipeline

In this notebook, Ive loaded the raw IBM Telco Customer Churn dataset, inspecting its structure and quality, handle invalid or missing values, engineer domain-specific features, build a scikit-learn ColumnTransformer preprocessing pipeline, and perform a stratified train/test split.

### Workflow Steps:
1. **Load Raw Data**: Inspect shape, data types, and initial records.
2. **Data Cleaning**: Handle missing values and coerce total charges to numeric.
3. **Domain Feature Engineering**: Create simple, well-motivated features (average charges, tenure bands, service counts).
4. **Build Preprocessing Pipeline**: Set up median imputation and scaling for numeric features, and mode imputation and one-hot encoding for categorical features.
5. **Stratified Train-Test Split**: Create an 80/20 train/test split to prevent data leakage and preserve class distribution.
6. **Save Processed Datasets**: Store cleaned datasets and preprocessing artifacts for downstream modeling.


### Environment and Library Setup
Importing essential libraries for data manipulation, mathematical operations, and scikit-learn preprocessing utilities. We also establish a fixed RANDOM_STATE for reproducibility.

In [2]:
import sys
import json
import hashlib
from pathlib import Path
import numpy as np
import pandas as pd
import joblib

from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, OneHotEncoder

# Setting the projects path 
PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
DATA_DIR = PROJECT_ROOT / "data" / "processed"
MODEL_DIR = PROJECT_ROOT / "models"

DATA_DIR.mkdir(parents=True, exist_ok=True)
MODEL_DIR.mkdir(parents=True, exist_ok=True)

RANDOM_STATE = 42
TEST_SIZE = 0.2

### Load the IBM Telco Customer Churn Dataset
Here we load the raw IBM Telco CSV file and examine its dimensions, column names, and sample records.

In [3]:
raw_data_path = PROJECT_ROOT / "IBM_Telco_customer_churn_IBM_dataset.csv"
raw_df = pd.read_csv(raw_data_path)

print(f"Raw Dataset Shape: {raw_df.shape[0]} rows, {raw_df.shape[1]} columns")
raw_df.head()

Raw Dataset Shape: 7043 rows, 33 columns


,CustomerID,Count,Country,State,City,Zip Code,Lat Long,Latitude,Longitude,Gender,...,Contract,Paperless Billing,Payment Method,Monthly Charges,Total Charges,Churn Label,Churn Value,Churn Score,CLTV,Churn Reason
0,3668-QPYBK,1,United States,California,Los Angeles,90003,"33.964131, -118.272783",33.964131,-118.272783,Male,...,Month-to-month,Yes,Mailed check,53.85,108.15,Yes,1,86,3239,Competitor made better offer
1,9237-HQITU,1,United States,California,Los Angeles,90005,"34.059281, -118.30742",34.059281,-118.307420,Female,...,Month-to-month,Yes,Electronic check,70.70,151.65,Yes,1,67,2701,Moved
2,9305-CDSKC,1,United States,California,Los Angeles,90006,"34.048013, -118.293953",34.048013,-118.293953,Female,...,Month-to-month,Yes,Electronic check,99.65,820.5,Yes,1,86,5372,Moved
3,7892-POOKP,1,United States,California,Los Angeles,90010,"34.062125, -118.315709",34.062125,-118.315709,Female,...,Month-to-month,Yes,Electronic check,104.80,3046.05,Yes,1,84,5003,Moved
4,0280-XJGEX,1,United States,California,Los Angeles,90015,"34.039224, -118.266293",34.039224,-118.266293,Male,...,Month-to-month,Yes,Bank transfer (automatic),103.70,5036.3,Yes,1,89,5340,Competitor had better devices


### Inspect Data Quality and Missing Values
We inspect the missing value counts, duplicate rows, and target class distribution (`Churn Value`). We also check the data type of `Total Charges`, which often contains hidden blank whitespace strings in raw telco data.

In [4]:
print("Target Class Distribution:")
print(raw_df['Churn Value'].value_counts(dropna=False))
print(f"Churn Rate: {raw_df['Churn Value'].mean():.2%}")

print("\nChecking for exact duplicate rows:", raw_df.duplicated().sum())

# Inspect Total Charges datatype and invalid entries
whitespace_total_charges = raw_df['Total Charges'].astype(str).str.strip().eq('').sum()
print(f"Blank/whitespace entries in Total Charges: {whitespace_total_charges}")

Target Class Distribution:
Churn Value
0    5174
1    1869
Name: count, dtype: int64
Churn Rate: 26.54%

Checking for exact duplicate rows: 0
Blank/whitespace entries in Total Charges: 11


### Clean Missing Values and Drop Identifier Columns
In this step, we convert `Total Charges` to a numeric float column (coercing whitespace blanks to NaN). We also drop non-predictive administrative columns (such as `CustomerID`, `Country`, `State`, `Lat Long`, `Churn Reason`) while retaining the target variable.

In [5]:
cleaned_df = raw_df.copy()

# Coerce Total Charges to numeric float
cleaned_df['Total Charges'] = pd.to_numeric(cleaned_df['Total Charges'], errors='coerce')

# Drop non-predictive administrative & leak-inducing columns
drop_cols = [
    "Count", "Country", "State", "City", "Zip Code", 
    "Lat Long", "Latitude", "Longitude", 
    "Churn Label", "Churn Score", "CLTV", "Churn Reason"
]

cleaned_df = cleaned_df.drop(columns=[col for col in drop_cols if col in cleaned_df.columns])
print("Shape after cleaning:", cleaned_df.shape)

Shape after cleaning: (7043, 21)


### Simple Domain Feature Engineering
To give our machine learning models better signals, we construct a few simple, well-motivated telecom domain features:
1. `Avg Monthly Charges`: Total charges divided by tenure (capturing long-term average billing rate).
2. `Charges Per Tenure`: Monthly charges divided by `(Tenure Months + 1)` (spending relative to duration).
3. `Tenure Band`: Categorical grouping of customer tenure into intuitive customer lifecycle stages (`0-12`, `13-24`, `25-48`, `49+` months).
4. `High Monthly Charges`: Binary flag for high-spending customers (`Monthly Charges >= $70`).
5. `Long Tenure`: Binary indicator for loyal customers (`Tenure Months > 24`).
6. `Multiple Services`: Count of subscribed add-on digital services (Online Security, Backup, Tech Support, Streaming, etc.).
7. `Has Phone`: Binary flag indicating active phone service.
8. `Has Internet`: Binary flag indicating active internet service.


In [5]:
# 1. Average billing rate over customer lifetime
cleaned_df['Avg Monthly Charges'] = cleaned_df['Total Charges'] / cleaned_df['Tenure Months'].replace(0, 1)

# 2. Monthly spend relative to tenure length
cleaned_df['Charges Per Tenure'] = cleaned_df['Monthly Charges'] / (cleaned_df['Tenure Months'] + 1)

# 3. Categorical tenure bands
cleaned_df['Tenure Band'] = pd.cut(
    cleaned_df['Tenure Months'], 
    bins=[-1, 12, 24, 48, np.inf], 
    labels=['0-12', '13-24', '25-48', '49+']
)

# 4. High-spend threshold indicator
cleaned_df['High Monthly Charges'] = (cleaned_df['Monthly Charges'] >= 70).astype(int)

# 5. Long-tenure threshold indicator
cleaned_df['Long Tenure'] = (cleaned_df['Tenure Months'] > 24).astype(int)

# 6. Count of subscribed digital add-on services
add_on_cols = ["Online Security", "Online Backup", "Device Protection", "Tech Support", "Streaming TV", "Streaming Movies"]
cleaned_df['Multiple Services'] = cleaned_df[add_on_cols].eq("Yes").sum(axis=1)

# 7. Core service presence indicators
cleaned_df['Has Phone'] = cleaned_df['Phone Service'].eq("Yes").astype(int)
cleaned_df['Has Internet'] = cleaned_df['Internet Service'].ne("No").astype(int)

print("Engineered Features Sample:")
cleaned_df[['CustomerID', 'Tenure Months', 'Monthly Charges', 'Avg Monthly Charges', 'Tenure Band', 'Multiple Services']].head()

Engineered Features Sample:


,CustomerID,Tenure Months,Monthly Charges,Avg Monthly Charges,Tenure Band,Multiple Services
0,3668-QPYBK,2,53.85,54.075000,0-12,2
1,9237-HQITU,2,70.70,75.825000,0-12,0
2,9305-CDSKC,8,99.65,102.562500,0-12,3
3,7892-POOKP,28,104.80,108.787500,25-48,4
4,0280-XJGEX,49,103.70,102.781633,49+,4


### Define Feature Groups and Build scikit-learn Preprocessing Pipeline
We categorize all predictor variables into numeric and categorical feature lists, and construct a scikit-learn `ColumnTransformer`. 

- **Numeric Pipeline**: Median imputation (robust to outliers) followed by `StandardScaler`.
- **Categorical Pipeline**: Most frequent imputation followed by `OneHotEncoder(handle_unknown='ignore')`.

This modular pipeline design guarantees clean, reproducible transformations during model training and evaluation.

In [6]:
numeric_features = [
    "Tenure Months", "Monthly Charges", "Total Charges", 
    "Avg Monthly Charges", "Charges Per Tenure", "High Monthly Charges", 
    "Long Tenure", "Multiple Services", "Has Phone", "Has Internet"
]

categorical_features = [
    "Gender", "Senior Citizen", "Partner", "Dependents", 
    "Phone Service", "Multiple Lines", "Internet Service", 
    "Online Security", "Online Backup", "Device Protection", 
    "Tech Support", "Streaming TV", "Streaming Movies", 
    "Contract", "Paperless Billing", "Payment Method", "Tenure Band"
]

numeric_pipeline = Pipeline([
    ('imputer', SimpleImputer(strategy='median')),
    ('scaler', StandardScaler())
])

categorical_pipeline = Pipeline([
    ('imputer', SimpleImputer(strategy='most_frequent')),
    ('encoder', OneHotEncoder(handle_unknown='ignore', sparse_output=False))
])

preprocessor = ColumnTransformer(
    transformers=[
        ('num', numeric_pipeline, numeric_features),
        ('cat', categorical_pipeline, categorical_features)
    ]
)

print(f"Total Predictors: {len(numeric_features)} Numeric + {len(categorical_features)} Categorical = {len(numeric_features) + len(categorical_features)} Features")


Total Predictors: 10 Numeric + 17 Categorical = 27 Features


### Stratified Train-Test Split
We split the dataset into an 80% training set and a 20% holdout test set using stratified sampling on the target column `Churn Value`. Stratification ensures both splits have the exact same ratio of churned to non-churned customers.

In [7]:
X = cleaned_df.drop(columns=['CustomerID', 'Churn Value'])
y = cleaned_df['Churn Value'].astype(int)
customer_ids = cleaned_df['CustomerID']

X_train, X_test, y_train, y_test, id_train, id_test = train_test_split(
    X, y, customer_ids,
    test_size=TEST_SIZE,
    stratify=y,
    random_state=RANDOM_STATE
)

print(f"Training Set: {X_train.shape[0]} rows | Churn Rate: {y_train.mean():.2%}")
print(f"Test Set:     {X_test.shape[0]} rows | Churn Rate: {y_test.mean():.2%}")


Training Set: 5634 rows | Churn Rate: 26.54%
Test Set:     1409 rows | Churn Rate: 26.54%


### Save Processed Datasets and Artifacts
Finally, we save the train/test CSV files and the fitted preprocessor pipeline to disk so downstream notebooks (EDA, Model Training, Evaluation, SHAP, External Validation) can load them cleanly.

In [8]:
# Save raw features and targets
cleaned_df.to_csv(DATA_DIR / "cleaned_data.csv", index=False)
X_train.to_csv(DATA_DIR / "X_train.csv", index=False)
X_test.to_csv(DATA_DIR / "X_test.csv", index=False)
y_train.to_csv(DATA_DIR / "y_train.csv", index=False)
y_test.to_csv(DATA_DIR / "y_test.csv", index=False)

pd.DataFrame({'CustomerID': id_train, 'Churn Value': y_train}).to_csv(DATA_DIR / "train_ids_target.csv", index=False)
pd.DataFrame({'CustomerID': id_test, 'Churn Value': y_test}).to_csv(DATA_DIR / "test_ids_target.csv", index=False)

# Fit preprocessor on training data ONLY to prevent leakage when saving joblib artifact
preprocessor.fit(X_train)
joblib.dump(preprocessor, MODEL_DIR / "preprocessor.joblib")

# Save feature metadata
feature_metadata = {
    'numeric_features': numeric_features,
    'categorical_features': categorical_features,
    'random_state': RANDOM_STATE,
    'test_size': TEST_SIZE,
    'train_rows': len(X_train),
    'test_rows': len(X_test)
}
with open(DATA_DIR / "feature_metadata.json", "w") as f:
    json.dump(feature_metadata, f, indent=2)

### Explicit Leakage Control and Column Exclusion Documentation
To maintain methodological rigor, administrative identifiers, target proxies, and post-event variables were explicitly excluded from the modeling feature set prior to training.
- `CustomerID`: Unique administrative key with no predictive generalization signal.
- `Count`: Constant value (1) across all rows providing zero variance.
- `Country`: Constant location metadata (`United States`) providing zero variance.
- `State`: Constant location metadata (`California`) providing zero variance.
- `City`: High-cardinality location category omitted to avoid high-dimensional spatial overfitting.
- `Zip Code`: High-cardinality postal code omitted to avoid high-dimensional spatial overfitting.
- `Lat Long`: Compound spatial coordinate string redundant with raw coordinates.
- `Latitude`: Raw spatial Y-coordinate omitted to prevent location overfitting.
- `Longitude`: Raw spatial X-coordinate omitted to prevent location overfitting.
- `Churn Label`: Direct text representation (`Yes`/`No`) of the target variable (`Churn Value`).
- `Churn Score`: Post-hoc rule-based risk score assigned downstream by IBM systems.
- `CLTV`: Customer Lifetime Value computed post-hoc using churn risk proxies.
- `Churn Reason`: Qualitative reason recorded exclusively after a customer has churned.

In [9]:
excluded_columns = [
    "CustomerID", "Count", "Country", "State", "City", "Zip Code", 
    "Lat Long", "Latitude", "Longitude", "Churn Label", 
    "Churn Score", "CLTV", "Churn Reason"
]

print("Excluded Raw Columns for Leakage Control:")
print(excluded_columns)
print(f"Total Excluded Columns: {len(excluded_columns)}")

Excluded Raw Columns for Leakage Control:
['CustomerID', 'Count', 'Country', 'State', 'City', 'Zip Code', 'Lat Long', 'Latitude', 'Longitude', 'Churn Label', 'Churn Score', 'CLTV', 'Churn Reason']
Total Excluded Columns: 13
